# Mutational processes as linear operators — 650M, multi-signature (self-contained, T4)

Confirmed at 8M: a process is an **affine operator** $W x{+}b$ (faith 0.61→0.82, EV 0→0.66; MLP no better).
This scales it to **ESM-2 650M** across **5 signatures** and adds the interpretability payoff:
per-signature operator quality, the **SVD spectrum** of each $W$ (the directions a process acts on),
**cross-signature subspace overlap**, and an **operator-composition** test.

**Runtime → T4 GPU, Run all (~20–40 min).** Fully self-contained (profiles + proteins embedded).


### 1. Install


In [ ]:
!pip install -q transformers 2>/dev/null
import torch, numpy as np
print('cuda',torch.cuda.is_available())


### 2. Embedded data (5 signature profiles + 479 protein CDS)


In [ ]:
import base64,os,gzip
_S_SBS17a=(
    "H4sIAMWtc2oC/4Waza5sOQ2Fx+e+C0f5dZIJUnMGNafPDCHEAIlJM+D9B3wrsbNL4jaoofveVbW3k9heXnbq49c//ZrH3//2+vPXP+v4eP/bjL/99tt6/phT"
    "/Pnf/7Ifv/zl64+//PWXj/SZUkkjFct15ZJq6YGVsVaeVsuc5WCll1qGzdRLX/Niglarc63q2FqW6mx5WW3ZbX3tzxKvzLmvkmu2UqqDvHHVVmbrvdg6IO9c"
    "zXLpreU86gPOXns1oDXinXzNZktjDqvLDb4+2uewZCmnUdvKvf3jD6kD6h1r8dGqY2QHc1sltZJXGaPVC84+Su/WWht9g10Gcx+dJbeRTKAb/P6wzzxZBHvv"
    "teZ2nhDYR5vWcqktLX/NGqv1ldeotaf8gCPlxn5kI0A9yf4KL7d6Db7cfVn/lNHrHKmmFliu1se0blYCW3MZx99z7XM8WOW53jj349KMsVSmDqLkMt2Wu6/b"
    "SoP9yMBKM0CeabmlshQZDvY2crKMqZk8qjaIL2rSC2o8PSbBY2NYm3O4vdf5MPeSMqdUCbBWRoBWs7w+2EgxB2tpq/dlirBUHnBob5m9zB6PL7M5Vye2CCQ3"
    "+P2ROemk7xNmCr19/ALzLJUlWCAsti58Ufok+h+w4eXJsVnvF8wdn/fGGsbjuu9wHSdNXHIE1jnPwLI1wjOvlRwi61rn1UaCjvJgpObixTMHxn545cg5xUl+"
    "u+cyzijsvCql+zmzbPy5JkUFke8RYTZYT08zD/ZzsYxv8SNeyv6+xJoJdo511eq2Xh/1c/VEjg7ogAAqnkkcViqsi2wmyzdYP1urMkwAkC71gvLW4hyNgHrL"
    "zpXwwCCR29s5fh+XDmKYqLLOf3npBW32rmVDRR5tI7PhVokcTGa7IPlC6Co08wxQ8QGpFLjNxCrflzJ/nnNJMTaJU14XzFYIVQ68JQJ/zAtCm6Q5a775AiXO"
    "Qfobnp9ruLlgzYUDOjDcyTcCtAHVwreT8AzQrEEo5Bn+snlBfFUJWr7bYmlwLVQl9maT2Q163hX9n2TumUCKZIJ5c7Nm+pOHAX/sk8zvlXNK095AtkYBIAxa"
    "gGap8R2SL8VxHv9lvZOFWeEIzIOzwdKQJpXGfHOqTyQZoThJhv6GEfd99WbxvSzzHGlTLm5TX3hufXIM8hF2OsS+g2t9doUf55hh7HLCcH5OorhNYqRAK4ER"
    "17Xx6qYCEGCa1A2YjSMrUQ5kb7uOICF9ksKSt40D2WJVS0GYDidRGKDKPmA0CsQIiDLAP62QMgciYUlNEi5bL27GHUasE2J4fcj1NUBOgd21TrXsHiHV2EEi"
    "I8Zg0Q8G5WfWSWKvACGGTt2B+pJ77Ms9RqRSOngMyiS3HVPJgWMS2zhOZPGU1KX6Bb+1N4yaPyAOjy0Ib1Fr2e1AjJwTjPrGGcFnvDFN4jV53E1KMqZwExHh"
    "FU6lt0ETZFaHFe0BiSzlkAjCQbKoi8GwVzzbbpEbev+scKWlOCEjGEkhow5b8UrJPieZTYkjUCyOTaBoxNbDlmnva8IDTUwb9l7BNPs4CNdUCZPLSeQLG2Gf"
    "rY5YWaVsEWqIh9zeQMkLnQ95EKANwoT6M/KYzQ1+nzjqUhecIKTq8cfbWK1JGKQTf0lFmirHZjmmByJNoZtx7GRCf2u5sj/58fUmK6UGVK7gDKjHYwQQmqDq"
    "mkqzr7WQzAQbfqP4jgcT1xPWLCvAIVnSFVJZlfvrTVmSLeKRRoggrELKcCg6HD4Y7kmIfo3G3wdVva6LUeKQe5SF1ONZygGUjibIw029PsrnVkfsSOViQA0U"
    "EUA0gooQPM/rN5g/YSrchQP5QJIgwAIDSjMTEQ6yIqQE75uE0Qbd4HfoGCUmZC+uCS1MuTfVWqoQUuwqHtzKMvBoRR28gYhPJDn/CxlUkal4nGehiLPDm3MQ"
    "KiE/OQwoPHsQwyycelliUVsh6fg7q5MaIbrfwL7kK9mNxyk+kIvqnhLdDXrOFYoWD3EO7bJHEgdmKLeqxNUaoBQLlUmsELVJ4IQVVfnKvKAMkq9DiVHc4Ot+"
    "qGBcqjwrzgSDk+KFqELNzZCbRF1XQlMviaQHrHqYHVqKx9HnEp9TvDDC4LdEUcJRKkdo+BT6Z0od4xocWOyCqk04NGnH9QGl0bb72wgwp06sLxqwsja4DX5f"
    "3oRzYayhRLIoCeL5OfEVOTBDERN2dVd2ErbUN1C00ii/M9IX3sT17KN1SsB0gyExi1IFnje1gC4dOVuOb0iYp5CYiAK1JVSC4ZpImHqtLKUzLCQm3Mep817M"
    "ua3jPhgTZxFh4p/pAkyvoCEl4VcUMXZF+0bLMdQ/2MU4c6RvVasW69TzUhtyoNtydUJRZmGsgtCdsTZCkrpO9QGdsV5Ol2ib23UPhpd4s+RkYBKnTf1Z3WH5"
    "CEvoQ+0C6+LE2EyAZccWFZYDsgCllXTaOUqPILwOYZrK3fNwTTpp5N9cbm67DFPwNrsaqtvzQLyEbEOW136CHUlSJBm0VT8vQegN/NyV+geC05bUDaeaYlcv"
    "Ly/q2bqWRil0eUNRNkkgKLIdqJA8W5YvFvRAUoU7YFwEDVGSlH0ZYWY7akACC3FIYR8+WhiqnrylS5yfWjwka/puWHnphYZUIPg6CmFIQ6tMdfrzPLeZr+Mj"
    "8Q51haYIU2fllEGkDG+QQ4Z/CVqkc4C68lHcGxrqtwjn5zlNQSg12Y/s6Ebqq5oJoh5q2I8TcBJzFGOSfR6k4U8pno58CiRvISvZ2A6yI6FqJLPcwPbJJJSw"
    "qRoxT74hs6lqmxbWOoUR0jGxfiEg12kmNgQXwQpUur2xPd2Y8hFicXS3IpeQcXQGyFHY+pwVCOoBd+CTtNtQ5SUZnNT870IjgOSAYgqZWzaCPNa8KO3pwTZw"
    "KxUfEUJId4i/R0MnYa2VQpjUuNCzqlqUvSUaC3VYt0Lv6tAfhZy2gtPcR72b23OSI9DV3mlcQll1rGqCIb3A21xmiJs5RuiQMrgejAQjbzQtiPcRFXgaWUEo"
    "uK2XNIGKOMRFRqSOJkhbKOC1uV2q/QYoebMnDwjM+QaqvaNA1RzYrusIPxH7Bt3eIbquLmRouie/ZcdOg7VEs178OnqedJaeUKG72O4888INxZ/VrE5KVe2u"
    "/Xi9qcOsqRC1DNU1PA3BhiY7UzHTUhCzFIga3xKdpCAoqmkIFH2oSr/alaJWvrgl9xhqTQlR1QtN/3qRg5fonH/FKIxt0+xBSqjMByN4NWqyaC7hYb5oe3CF"
    "bHq5MDyqQONG6SqiOwV779aOVNVAlVD+GJ/q4SAmNCAZfQq/QAX30nBgNG9N6SiPsGW73ly+XBe2T7iUXezajgjwSQntvxiTukA2BYZOFWVXUeF6QCNvcCIl"
    "3wKEXrVX/E0rcu09rRgHqEOXEm8xziDnid+qCZY2HzpjqslFLmJn5gfU5JRuR0o0QOQwnCL1wHrcoMtCBDlZ00ga/J/aFTaIRVMBFrdHvwRzsfLda97eTy1F"
    "0mgI18ekkPNi9WAk7RxuT8peMlbZStiaSy/AKZWotoASFsM+KXtODvlJ81UfUM2fammSst+PJ02El8LI3hz4uoMrKWLWsyMsdPY5ZOo/jBSLVmGnZ00m5Xt1"
    "m3oYDZV5O4cSoPhUk3qO2U4u3IGjZmr0huRoG65G1AKoAaAiVm+P6TCRbRLDKHg3trGppJX2upiqw2RvZFd1U1+/1ySp+aCjo7apz/PWh2LDTsk4darrDZSw"
    "VU+oGeIBm+Y8apIUnbtJer1LQnQQXMXL4eYS0JTG2mXdoq+jqFRNOabY5mJJ2pcDo5L5MFUjRR5Mcoxb0ojftjSnjS8a9fqIX40HEUVIsMASoC4lCF3qCKz6"
    "gI2NDqV7dkxjqn0roSlJRMkRhXxokpBqr6iv8RZKr+KXjoSEXw5iDi3KK6hxK7+B1JGhqYu3DjxOS76neMqv+mbwpB0fq7GHyGjWfbhS1cqr2JbpZVkTIpIl"
    "SdxXdT4Xk35EuzQXkJpDUW9YrdLVLb0iGXvS0E/1YKbyBlIhTWIvBqRDI2wact7KOb2DlEcoha6mPdPgriE69Y667AZPxmk+IAHDa9Q5OMbxVOlXAsG7PiIU"
    "BZGkQVr3PhpMeobyqYuMFs+SBENDn6rLrtcjFFVmqrQFUaSeN9r2InWHMuuFUhtghsSVl7aj7g00DQClnaMb1dUcXyTWdZfjBs+wsamJ0QVP1kXIgWjTOMZM"
    "8AxX0F0Df1xsu6hcSJdA2uMZXZMRs9K7g6tVczPHadASQY27usRIfzDTcUzdKHkoEF6mpk4kUR4M2Wm7lGQPj6kpIgpPosBNuSShdUBTo/ZXaz4WEBENEaf6"
    "R5/0EBOcBjGrTsH7R8VJUiGeuu7p/ix5xs4JDcOz29atbihpXerwiYZU0XshkNEfOIZHXfApinalILxwZXkDEdy1o8KjAjWEKEyoEq3rEjcYQw8NavCLBhEt"
    "5uRFsxCJPEK7unJMu6mbtek6o975xu70EBVDVc6rMaKKg+Rsy5YTbvD1v8qpwKFmm73GfEOzCSoWqoEO3tN9g31PD5FzEaVTnVjRPrsGym7Qx8S4j1cWPMyi"
    "jgvUeHRN0DXc86gmjXcbgAXN/S7GCSCZNPKb/ix+MbVlWVzy4/t93vizmYDAouQWu4cOl9CkJynSJyThOziHaE+7C3BJKoylm7rl9lTgVIKosJ0+qLr4Apxr"
    "z9pUB5pj6sXhG/gs+ZXixrJIHQHZeigFU0MsMdU0hdic/P1/ho5dX9bVEWwXk8S2pzumSfkcFxRvqv1RC+5P03mb6FGZ0E45/X6fOaq3rKx0QAlvY9mlIRHr"
    "libt0S8RJ3Kg7mPzG9Z3saBPuo2VhmVjD0gIgm0vso/+We0d9DoiWauaHP0uYV8AOKeIc6aSW8UvX0wT8M0WZ74gMmOl+qaaELcU8/3/HhQIpG7pHkrMuGLc"
    "xYaarvA0v7miUlPyvKmPzL3g0I0++rOrmrjBuJzR0E+RpCSJSzk4kFerCmjkEyBiha9IsUutX1ADU+pAt+iLNPPXoERrgaLD4PVdFSNrDMTrg0PmLqhF2hIK"
    "DZdIzw+NqfSThTdQBzjLkwoaIRiVP2kEYSf37qxRbaLppoFea93RsP9aBA5gtzlIVj9kGIh+XZb2B2R5ujlNyW+nlbLqm/STBQK1usEvtUUUagLfNjuv6JVQ"
    "R/yVPVo0VUt5oztWU61zCSZQl12a7rDJC1IpeY6FIXTWTb/v29rxcdGUmoBI+Y6GadhQRlU/dYhfF5BURBC1v9xsEAj/qw7plxMOopttqZdSb9rcnHuQ7kJX"
    "CewSX8bEf/eHMo8wuBJIXQvcSQihxOK+SSBP079SiqIno/Y2/V6Ac0arDAy+DR5396a2amr+NwPU8og+GMBGbBpIorjuqcA7mCRAtfDb+WrKJ/1RNT93gycL"
    "0Qu6+KJb0bTcBwYahEzxw6Ks+GDBNCnFFpqre8u5J7q7D8V/wx9VP6YuhsfbdFPeGZANumGd+llFzGLUtFC2dBMTEjPrnkAMDEGUBzKNejWRiFk1XR+sr/KL"
    "Z7Kb8lmxbszU+5uEjncXIgZKtcZZ6em6isShfiPj98IHIyLlm+nz+mz7eoWCLuZe29bX/d3Wkh21jopox3QV3tqeZCd/x/5Bku3BU/yGRtgZQetGPTACVLOV"
    "qh/ImNs6ylL3H0tTaP59YkxKQDK7aYx+RmzwNBSFuEz7p04Bdd3CxP0zb9J18tQzdGFu5PjpZ3dAYEMlSqeYXZurBVXmZF3iuwwRpsEH8UAYtMDkPP3iBS0b"
    "trajihoZXf/qcvAQjm6Iq+4E+bD5yHZrlf2TGfZqASGn9T3NMg4kcqk7eqkq24yq2u92wRriq6TphyhRoTVoFZlsur8gQnTqHpc4uWDZP/HQVCqN4mX7TVP+"
    "7B5UeTw0h5ZIv/SjqaEm2RxNGfYGamSbd3d4Xwk1iuL2bz/c3it+t7Lnzerw7kWh7vwQ/FW/a+ozJkIa41FnqLCw9JWZGjRRNPXTpPs9ntOPC5Tn061597ZJ"
    "TjeCpIPPddv+mZt+vUDg9egokA4UAlN7Wy+Esa7L4NvTKvc1CGu6pK0//gN16CiSVSkAAA=="
)
_S_SBS2=(
    "H4sIAMWtc2oC/72asY4utw2F63vfJT8kSqKkxoCzxd/H2wVBqgBpnCLvX+Q7GkozBrJb2vC1956dGY5E8vCQmh+//fU3++f7bx//Lv3H/fO4fv7997l/yOn6"
    "6b//8Z+//v3jl1//8euP9hopZe8tVy999n/9JXXAPlNqM81sM2dfoL8qF9Q8ZvJWcr5Bn300q9XdNjjd+px15MaTBYbBjx/plTBYsVqt15ZzbbbB1lJtxVIq"
    "nccE2PMYtY2Sxqx2g9O8ztJ6Tu08U+/ahpVqZmHwzWKspNy4Ls9pTa/YAIvn4amX7HWMvMD68pZ8zFGNN/C+wTpnTizc27VBur2ViUVLc/isRWAY/PwxX30k"
    "G1a9ltLHZRCwsZsGNBJPClA72UubGB3Nb7BmvJFnyz4CbLlNqzNx3czH3hsXmhaYvFb+8FLYyy7QPPfOnszpesMF1t6sdy2kjZRvMI8ybOqHvsHZcYKXZoV3"
    "ExgGtwsLRsuo2UaZdWOpNzk1tV7G9ksx40nEUvOU5gOs2Ku4sPfzyCG/JpZiuYS994/8asREZccx2Me1I4AEhdnAKO7OARa2rBBSZYw0yw3WRDzOkmqqATq+"
    "46USEdivqAiD8mC2ZDwnWbFuM1yQyYmSC2GXR90gr90Lbm3D3csNssHDmvx/+Xq8Ri/szeSBfbofg5+40F+ZF+lenbzwdt3hrzTncpcTqbkHyBPXNpBC48ay"
    "/k28WBo1wFF51coKJ9c/7V0ezARiwrfNWU8OqDkZgV+SE26BdVZn0yw5wX5jHZ8QIJb3dUSmSKMnEsrC1PtHeeH9NGtli2b3KxsKLoEwMGQKueuN7TVT7/h/"
    "kjzs0Q0W9t2By9i3Q1bGxaWTYfmxtM/rXdgvbuENDe/G+7HQRIiLBHwEhNfwRc3NCMSDwRKKqk4UB5ZIhg4XTiAt7XNxJ0k5k4n+nDwr2/lVkYAtti1lD+fP"
    "bvBP7pBTRImwDCmUSa5AJTtKoKiEH4rDoSOW9nlT5xC5NF7cWuKaDbbUa/EBjzQbATrbCPMRw/zzAEk8UjThrshbYok3cxgDtug9DL7ZfBiZ7YciHTbYbuKJ"
    "REythGy9uEggsTvn6IbBXB4gr1kcb8dW6HZSCi+YwtMeK/z8kspIXiu98bCqbATMpBNPx3Gkam32AEuZrq0Y7QaVQKZotxpU9vnLx5V4ndQipaCCKymF4YDc"
    "iWh40Tfjd2IVbqKq5RIuXCAvRgyw6L5LC3s1IX8s4vJ6FviBC+2V2VGnbrSpF40twf2KRbafzNv7lLEGmxGM1VO+waxyyP28SICk4VzB3dnv/jAo7lz1dGQT"
    "pybfjJj0GNER+bMZEfYoEA6px16PB+iQWZveaju3s5G6FfLI9jD4GUyuuCG7YS4cljdI7LfWWBZEd5eMopKxCLM8MKKEh/BTv0HilKRwgrgue6p+65cmLmPp"
    "8KJCbYOqLiSGNnKHOpmA2JFKmKS13yDBRV65+CjATASyaOiM7d4GP350yIdfshm8jkUkAKrIEPm4duYcoLRRQqxksjOkg0AySnvAD1FGOiHOG0wjk9FOZ0Pf"
    "KwnT4lXxHU7XU5QTKIwGgRIxc73Dwqp4oFdtaZs3WL1PUU1XSlwgsqOldTP6SmDY+4RcO+EGO0xeiBq47gAkLpQCjiPmDJDoZJtYiNjEbpBU8oEJYuAGSUiD"
    "Z62O9b4/P0KCXrud5EHIhj1sG6LaVp4CwbY+NliUcYjBLIHzAAlHJCjcvXUHyT6kLuqqX2HuQ6yNYqDkV6JlnuJe+CvkOKQ+DpU3cRmUyObnfJRYgwBUU1VW"
    "8waNbHflMcx6VaSPEKBf6jMolMuRnKkEU8Fe0DhPFVvVGyT9SLa0CsK6mfUlEpB05bmXOvsI+bkWzy7yCElGEVNgZCXpUUlvsqqfC+E0Ug2lQDV+gpAGLG8H"
    "XKqbulsN/hvL3nfyU0vgOji7jBogTKsCXyVNygPkmjImnmobg/Vs5TPbf6/vqE+SjRhFh1CLUD4BUpIoBWgYitwmGK5kj7XVJNbsB+RN5Gl+uVmg4AmYu0mV"
    "IBbC4JsOgEsRkOI/BV+0BRSHkqmErkwsoUtQpKQO3ZNrOw5IDEM76FsiNG4vrqojjpOAPQGjDHR+KSFCNEKYaUZVMio/u0nAwI0lQCIRjbNAPxJRYGrqyTC7"
    "5SAUx+vSTKknG8fgZ2QgOYLHoGwqAAL6wrgbCSGhxm01sJWqrDKLDw429VcFbxB2JWHInCLOQG6Erd089BW/iypJowBFGqpV6jBzOVfSXqDdsXCndldFoxOl"
    "mvTNDBl5qsahKJbcwuB7/RLqSnprRP1I8RST9qLcdvHVvB5iIkRSIctizwdjHxEbWIzFmSLYdZ8uq2ErxGdV7cvoND0pxCeBVkTxpl677utMXJGUwz4PRlsy"
    "JFbrFuDSBtLEUvdTtj4ftInGRKjmoYjdbRhaS/0uCU1sHXBV5CQ1LaH2ANkFBDubmc8zh0rblOJexPmQn6h2GYQKKKNbVEIJFN7sq3kz28+mRkKljbYIpXqD"
    "1BBCVXl63le9J7kBs1Tk50fIz2jrp4ITETmLRf+hAYC4Go9KquyoaAoC0pxoJY4fILkIbI9JQ5cOwSEEVZj7WnxKEhCpqz/bnKU8YtETrxKBT5CUZuNm2yCO"
    "QXxRVwyBk4PJLvF5sQ7VC8aAV/BV2htKWBF4cC8MPyMVCioCRUPkSYeOG/QVQvhkj0LYLV7YNdloOfcwKPWpFm3IFKu6uguBRdLJ1syinl5MmpHHEzI+b7Ar"
    "C+nu+TNOg8YuT9hzKGuCWC71+dWWqrsiRod036Z8W7HvuJk3f4Cq/zyaZ4xTWobGMxLs9F2PLd3q8/+0f+r/1ERMvUg6gTALXQyFmPROh0gWSMiqWa9bfmrq"
    "1fSOkGovy+CRnyhbJRfpSUjt+M9t9Zc8JGtJG+wSitSnmo/SXODQwAVSe6hl9ljvP3N1D4Mf3+prInKJ48ME+EUOY7+yWugbhJpJFbi5niv7mktU8X+1MLdz"
    "UESlbq1rczZGJMFrsxW1cudC9a8dL/Y7LwWWRYqFfwKUPPMlKqHJEfY+j57DRwPeyGKgDXJtQZpIDaZNMjkvyYx7lARP0CR36aDmLTBH6qskQqY/36E+x4tu"
    "zNRYZIXCbsOb+nVeAafCSQHmVYbIPlit2AOUDJnSOjNAdlzvZJKuV5cXBj++ygkKhjZpFCXv7lwpPMSGyiQdV7lBpEzR4NFb3j2uj6U7KjHT1pVhcLWAolAV"
    "ly5y3D3clCl4Y/CbPT5jJ5rqR9N7lxssrKQg3Ou5nUCA+1SsbeuXdyjQpvGZKBpz1BuPtpjOFkEJ0ZikQYDZNZYtkgIl2kuBVAT0DM1eCtmt21urGu5Kufsx"
    "+F5dPOkL+Y6hmaltbUT7KAFCF5HqUUFoLio5wqb33B6g2k8Id7dp/iJY9VdKGq1zedjblRCVDiFDkw19eAqq+nISGW/kfHpOpB6xm1RW87xBiIoe1agOm/ez"
    "Ig5W0/ghXPj+hkeLiCetyZb1iAT6Stfol+TsV2twgdKe4uh5YmZq0icSJUNzPjFz0jCXuiauVFW1UWdySzuMLhQvbeWdpbIpcKaWofUb7GoCm2am/b6d15pr"
    "ogvPvEOCflkp5Kaxuhnvpzlga7OyMM06H6Au5D+RAGqTSJysjkcjmzsrpEPzyzRPJs07CmU5uAtEo7noFckxc4AaIGhkxlbZkuMbFG+CpLxvJ5nYGvm/93WW"
    "Evb+ZBd+riy8cl4dYo0Ipu/QgQW1kf2YJzGprrOohWDJ4wazmkd+UWJqu24upLRUHYkROXHJ0S/bQNbFPuN2s3rP8wgtHTDwpHKDNkUZK80PSFNATYK3k1qn"
    "tb6HHMW9ciKFVuVhRyORWDUNgJjx/dZ8eNpkUO3EuEFIW40WjdwRl02UpjqPXMph8K2+Cqqk4Lr68343W8i7KtlCjR17IK8Qohlkl0rJN1gUXJr7xiTbX4Ss"
    "a5/XuMsfW6o+0GSQLpcAZonboBH6iLCpSe5+NkVKfXZamXFjrnkw5TK3A6qW0B125HS57X382S689GhXN6HsNJjTQlDCpvA52gsnxqpRmXTLLG0N4Ws5IGoA"
    "eSg1GiWlvHR2V/iN5jlmjxVuNUP/IoGsKbsdJZh0ykJ54kl5TxAzOU7H6NrYuscEC1yHeNrfA2rXmqszgunD4KcIpUlKSllMi1myQCqBaxDYap4b5IH0ifDB"
    "LPWB9aY+TWJ2gxqyo1dNLzXOAu9pKGvTUEl9F5yxQYqpxqddSu8MzlD+LkHqLKE9QPRMXbxfbjCrJSxDexIGvz0L1EGApvOk2Dn2qyI0MTpBWB+Y5k/FJKBP"
    "h7YuVa+fz/p0mtQ1+OEJGUr2PXZxDRJcIx3NygKc4jPNN2tvka0CKemEFXWWmNygBg8967CuXvOSMPh1U0g5KJonZZ15bLDpgKGp5xp+rsRTRcdfXjydtkYq"
    "G7GulufqFH9+PqehrtNlWodW1+Rxgxr1kgRoF2prgGgbVgP3rAnMA3SNTvus29c679QMgtIk5giD3w9kNAlZz+77OFBdMLpKx9ZZ7dAD5A9pgfrfWF2DG1QP"
    "0nqGvbe0NQSk7dDBTd6no75GTzSlHcbzmIq79KUr8z0cs0CdasqrefWE11naEq2Ed9OIcrnw8w8D0aq60DUE0rhkg0QN25p0gre3mQRu6zxM48RiN4hW0Okv"
    "3V+sWqRGGlY1ygTrMvj9gbymVmRQW4u5ZAsOrRLumkHZA9SoVsfmfvTNUP5qNFXtRMx7DbSJe6NUaSU29kB7aDK5BjJQ/rXLwXQ0tUjWuk6uywNcDZn03vag"
    "ukJN78j7sZeneWhe4/ApzbDPvAE7F/Xl9RlTThJTB1XQqrqrQ9D6xGLoqNquico1D81J35/omCpdB8JhMKZqWaNBNSvomZg7aR6Xr+9M1JddGCXTpfr5wVu9"
    "MR3vSHXGraYOB30gkihXMlyT0KbZKwGQNLaabSHkutoLk3ZSQIjVlPmuyUHfQFX7oDZwLgQiJ8RZqsEUYWBlG2+XdEygttOvqINJ6IM0btdkU0bFcpoJNU17"
    "20GIPnIKQq0LEctqAqnzJAsbq8ohrRXu0u1i8wsa+loFCUIjd80bad7VZyXNsRHCG2oSeRJQVx2kY64a+lMrpXPCjNwi5UoddH24s67VMJJNJmAgWJfZoufz"
    "4r7kil2IpkWug+hyAersiT7ejyTm+Y8Bp0bLGiCM1YzUfRShg39ytOzGSUeRmqK4OHjPAQQi1ppk0iHM4joEzrqZfcthbdMgcYSYqNAbtWvnBhzKcwgB7fQB"
    "xV9NGqjeY0mBdNRsoVj23N41YhgavZYaBvVdkhJMVQg+nlt1Q0g6RVTx831UIJ1XksYOSI92g5rA4jz6mBkgKQjla/pp8aVTmDuDlT8cJW6wark0SOUGxX/6"
    "AEE1OZX2AFmO2gda31M6FLisTjp6LoPfqEn1vFlnJzrMP23Mkj46WqL4zBtU4upbs8UIF6ghl4YyePQ6DQuDH+KQpsR2fWgy6og9QcZIgBGTaX8/wpXDFRym"
    "UabPG9TcW71uOsc86oIkmPo6KzpbeqtJlc5SdMBa2xE/64SiaOaslvQIR31KoZP3cnZUnMEVrunk+WIqK6G7Rtf6aCbs6WRWH21pEg5dnC9pEJ3QaVJtanPz"
    "rvQsSaUpllV7gOuldFS0vx2Y0P0ae4y8vpiI9amO0XOyNs1cbNM8UFGNHvQfbOyWrou2cEgROZcbhGUlXvUZ1Ab1RRnEhsyuZTzM6WA9ifwJA6qk7XP1bCrp"
    "RBEvHdKuv9ZXGJrYaNw1b5Aw0kHVXH3KdftmZXT8rA97bx3aVp0CqSlDAJwzX2WDdG7XGdH+9GuqUaeONW813yCbpu/6dBK1P/0iZOFc/lfjE8Iw+M0Bg2nI"
    "3afOa/v9zUqRLu/r47kH6PpSUHN536C+UNApUpJOWxnxP4wchDESKgAA"
)
_S_SBS17b=(
    "H4sIAMWtc2oC/72aO49suQ2E43v/iwd6UK/EwLqDyb2TGYYBAwacrAP//8Bf6ZBSG9hJdze4MzWnDyWRLBap/vHrX37N45//+Pzr6991/Hj/bcZvv/227o85"
    "xc///U//+cvfXn/+5e+//EgfKaWexmqzpl7yqm0ecPQ5esuWazIHx8wzNR7NveZ1wWbNRl35fHrl3EfNs5VaUnF7r+ePuczF31sds/NQgH0tq6WuYrzwPlly"
    "KaOUxkvfQJvW+khz+cpyzaNay9VyTz02+Pmjfcy8euLBPHg8/+tPqQGuYSONnPXBYg4Wrdt6Lpb6DDCvNQsf73kOB/tHaQ2Y/VlZqQh0g1/Paoy/c3hl9c6a"
    "1gF7NbOeVutrBFjK4tVlrFJj24A1cci1z7IumHSes+VUkuVt8DNcmK0tDnVOttBLuSD2Bh6Y5ZxzLzildf1ber6g9VpKqfhxBtjWyGktTpUAcYOvH/ZR6xqJ"
    "5daSU2v7TOwDJ7GXktvodVUHa8UrbKHl3ue8oI3cGoExWzxpVSdN+NXKYZ8j/cSH5YMzmnIYHhyGwdwFEgaDUEl5WZobzB8LjNjNBAlRe8E6cuE4W7PpH9ce"
    "2liVwOmWBbpB9yGHvXh7MZupxJECDjNtdfGpEWC31BZbLb3YuiDn1ju+4qQCbInYwTOKtbENfoUP20hE6rTayLB4DeCY7IXkWBbJxW88Otpq+CcfcOJYktAm"
    "SeCgrUyuGlHNuTa39/oxPlLBrYOE7Z3U8bBehQSYbG/W6amiJ7UGDiOxinnBlazOVLUIB3kfoVlnKbPXeVz4hQv3aoDX1BFOTtZPpLA05RCbUcg52DpZUgcb"
    "yKTZBfMgTgcsMT3ta5pNL878iyvd3lfEf1dSEB6VLG8HnJMo7au3wyjWZiLx+ctIeb6B7IUE4qgPxkMcatHWvjaFig3Wc0SEQZpBEfxYiMPKvkaQCR7qnO6E"
    "kAigC+aCI9nEqC1Alsn/imOOxs/y63Jo4Qwna81i3doPaAWCzjsYAmNhEB2+hbnzBTkawoVHrQVYlf0lA/Zj75O9cFhNMYS3YnsNXsDPFV6sdsAF+xKAZEyv"
    "44Kp1sQCyI2INOJcUYqDWxvzbXtfEc3EJ/yTFfkl4p4oUTrg9mnT96xUGSIoDrWkcUFosA6Co4bzmhGxUBdHqpTeBl84cH1A1LWW0WE//rqXuD4I4ZTbUjL0"
    "3BzMtkONOplgywMONkg9YzN9OQhTEQ+mEkWtOzt8HQfyElg5L/1U1gGNUgyFEeLhliYGWWNsspoX5IQqZA9vRy5timQx1O7Wptv7nj833UDu7N56gFVRq7IP"
    "ubcLQutsGHxagGQVgcXOiYTl/CmDX3HcBAdFnbI1a6R7ozRUCYqkhDuFcY1mhTgqBqNdkK1B8AqUeJIT5anJUxRv2wajBmZjIYo0PoCPHowloBIkDKyOwGAS"
    "NkIxYEv1YE3GapOEcCxLDWCwEv3NbT3ew6+rKzQKZW21wHJStELx08yxRjoSd5RtZ3QgZSIHnAU/mEQPuYtSIVqKm3LWFIXhCz3dZhRscn+RqmMUMa5jq8KQ"
    "lBaSP803sCDhdLor1BklHb/vrKGaxTFuv6myNHQBMoUd7hzjoCAmShdnmZ/3UqVZVtmU0OqBCFAYsmGtPJC0AxEJGVlvP1/vopMYRxdIrYkvnM4LWpLw1zsv"
    "Y3PGMCgZXdlrKxecKBQUqq3IHlKe2JX4wuXZ7UmwQH9iwYbmgDddcVC7ckZGcgTEn4PKtKIw4nS6XZBAGgpkeCJAgkgiSRnVN9m5we8TjlqCGwka1GhoE9QS"
    "ayWZ5Y58QZZLqPCCFQkH+8KvA49afVSMG4xyN1UsoRWIYZUobUM8b4QXlaeFWCZDKPyFLUJP8x1ki4u0be2+EwlLXSMzyrPDKzo5FRT8gvKJuZCSCG2KAzKU"
    "AA0VA8tBS9R9q+K4NxC+JWtInVjvZMuZOFMIzuEG5cMuSa1UoXOwkJJNtlBpiGV26WBHihGxkphXXwLCDfAzfBWfRsZVlXiiyso6Lvz8o114NKd8N7tCWgw1"
    "zznRiZEU6Ix+XLislE4B5wyb9XeQKlmreOm+k8q2lMaIpG3waE6TDsKHEOY6IhfSkgwaSs8RkYA3uggGB9hRb3KRlIz6pXnBggpIeHaoc3ODr610iEwSHhXM"
    "ulzpiETVt1BmLSQRhNclL9HQ80IUbngMTTXqaeOI6b77Q6TCceBRnAQ+e8cVqJQe2kAaZW71B937yfFc1UngLwpwvWBNSM4hYRBPcsZT7xxqMsJe+I/nlt4r"
    "bb6i46rqGFD1/JNCqrEe5IEE25QCfgez/oPB832nPk3kQazy36M71wd/oF/i7/TtKRSKzgdNoa4Zz4eWGeJ7WjHSebULQop0C2x9dAdRTrAxTK4oipS4uhNN"
    "tdQpojn1YGCwsUQQbOxRp66f8oqGo8Z5kRVGiWN1Kv/dMU6HTGLjpMR0W+49SAH1IFpiS6EL6OMoR5ILRH84RWGgpkwvGWNdUK4GsKMgKJgQiWIb4jC393gP"
    "qYErUFlqwZyk+Cy5QAVXpXC9QlqA4Hyk1nTXbQxNqPlFnoERwqwc37OPsW29zsyFt0lyZMJ/hBDqijhTm0UNivIHkcLiULvmIKlcEJVDaaVeF49sNpaUyUOT"
    "m9rc4OM4/IC9IR5HfHTHoOQtD9joet7Mi4uWS21kMQdjF3iPVF9+usgq2Ia2l6gsbulxGwdAvOYdWc21VFUNwhNF4eX9hqTZZmmib/oJbAzDU++ugZkkCynV"
    "YQNzW9tlWeMRyHl79dHQVDE+sTRBoXBtBCKCNVTnYd52IE1ctnJ/kK1SIVpIaDxJFnWOqF0QJ8lLf+dJTSIjQCERgpOa8GCKUgoq9MD+x8GmyhzNJcXBP7sb"
    "UKonoq5Vt7XdVCU/CDQR03g0bM2qK2UrNiTYhigFKq4Shb3XgAabpHtTT/hAFFv8ZpusupvZPsoaJyR6PJzkOQS0xZj6AVbxQBRhspv4UgUKiLhmy3tA+ECE"
    "7xJNdopFmJF7YFI1YZrpqJMT0lIuuwmQehSgQZAldUE0GAdRT4E2p6ZuBCLDUVkDnGY/P99V5Nqas6EXiKxQkSxHI5NM0s0jDtduPUhmzM1sFzTpcxgMjeig"
    "qbWFVa3vgvnpMrJ8EBnobkRTH6bhUlJNH0p7jXsy3U5zkGBvPKbYgaAvqGKFEOMtw8FKcYJL93BhiHC7G/yEjYu4mQcMKhvO0NCO5IbonSbG+2koa6gfTJsn"
    "DoiO04BxE9AGJ6o8S7NqhGfP7M7tRdvGX1jQhBFUuBxEEquDMW0h6hrsSoKLyJbmIhckgdEqVTOjeCckg0jISpnxuFDZRQFXUwK3aBh3irrkD9oNwbhWlH+o"
    "o2WRP7I71zdQBuHHejQBdVlFAIIa9kxj3J4USNOrlyIu5RSzFlORIYSTkmi4NpyqSHt0oS7wglnTb413fZaBisxqQijCHae0N4Pfq0jFp6KFFF6uF2kIu2IZ"
    "gWwpX5AaOlQaWUxMM21PvRYd43g05KdryO8bfdMsfxDS6bTv5C8ynYykqKwLQui4bkiWnNVSi8QLGmb3Y/Brz77IGiSIZFG13jzwknQUQkM81eeZaMGGRSk3"
    "S3vDds+haVQJkHxuGsVR4J6Td3txfdA01iNmqBv16GHigMRVROKko6fGyHTJMJsmSm8gBXWo/TsY7NGEN3GN23MZsgNf0kyrH4FN3SfQu8DDQRykDQ2sLkg0"
    "ahoHzI0uE1drYuMqhJaTGpV2i5SH2/tDHRgi8vdnX0jtqcl6kha8erHCi7DPzI/YfkAOD3qqlKn4uMYSmkRunRUp+Da8JFSolHBjFfPHbBwRiEHN2tmo04ku"
    "HpLym+M4F0CAmkYO6fIZ08tKN7mXpga+xQ5j8oyPyCBYT/Q2DkjNE/encjpVdrLoOjiTJgK/4PaVZkT1zCqggqrZICk7wuDDolkqkvdAWfScw7FVcMKkNXJp"
    "T7mlOnRN4Veo1o2pN9covsdzLLFqWsU769qWXg990qxIbpEQm0me/NH4XnNKmQxQk1RUTGOx7TItAmpKaVJvm52eSoTD73jjYUM3+PIxm4q3Ol1N+HxUxpLV"
    "gSv6emxXdyK6ZdDI1qc2wkxqMq3YmdT5vizTLNENPU7Lu5AnVQEV5MCm6CNprhqnQ/O5b67QTFHthbEvXqo7vsBoPRsJ2veAy225v37npkCjXl2wZGl4V8xZ"
    "Q6hJpGnE51pcWGbnSzlXfU1SVog23ogWmdvWGZuo+lNeq2ZULcfslopFL0AVt55PiCn6EVUEKNLnDYN3oDCcfEb3ungj+J8O2+258i+mbSjQWfzzAfTS1GQV"
    "VnDlrYYB3a044jDawQpZpvHn9BG1JtMKWuQWKbncknvMFPV7zIDymYGR6ewXg8vpT/wuITR1mTPyxTQ74g3VIoRQUEvyYmkm5LbkMV2roN4nErHvE2TRyFTd"
    "O8Fj21lZOUjuAdEa5kAqhZq2zexBYBrd73AwSNufXy4tsyiTOoDbu+aeOzM22KWRaRWajwHzh6ZYJsrWxGK8gdLomvCWAIvoVKTSnD7d3uv7YRPvgJZ125yC"
    "4dVrI4W6tHSaF+STZPcQkRxwFd2EPJfRm/a/jrKk9lSdg67S7ZA5dYv0R99THasLxsYWdBklZZjtgmo8q+6MvOw/hTTtqS4ivg1aJjf4JFlR3S60C2QWradj"
    "c1+ISlhn52Dyo+jOXeOC/o5lfVtA8ed9FMun0Ig8NXXftu5sUhf0a2eLbkguSG1spklk3Lno3iZpNkrdyU5eDuoiofQz2EJj0J0pIfbVpBt8SQVhZhc04mWG"
    "CmK5SrwqpRaYaWQgBtb9lB1Q4y8TOYbUBGyKaLTQVDU64aJcy5KxU4sgg5Z7D1DzHUWLOsvmoK6DKcd7/JvnBUnTottVoslBTWo4H80fx6PF3ODX97zfUuNF"
    "pkuT4cWSYgVG/Oxe2i5mutvfAyjHKDtF1Yjtryf3zlgSpU9+N7hT1bFcsOtLGBLxZ7qZEGa4jrSscQ/2YGroROfnQSGqM2oxw94rvuxS1WCj3MiYUBRwmL7S"
    "QL7rqy3xDRiJETXVeDvFwFTgnjmzk17ukxPu4jQhDRtuUNepRZoeTaeyk6JZQ4Hu8JL6OJpZbVAi/9RkvYNiFtMseUUMcZpog6mZ5HpjlzOZVBHlNAkO7Mbd"
    "qch67Is+KZW4HCJBERudpUlKXVC3dpIVK2ZpUPPQwIOSyroLBr9OZw7Fra05mkjTw4ByTySiBhQbHi4cOXmK2lKxaRdLujpsNSa0YDie1e45UnNbcZdK5SC2"
    "8NP55lDRTf3uY4nb+EZL0ViI0yz6plNuFxRF4irawnOVqiH1c5O2kFxfria/bejq/jKRfNAClODr+joR+75Q1WyMRrH3dUDNG5OuALc02zz9piV1wUq91xwg"
    "hUqULtLEXXfMcddJftDc75H4DBEmTByJi/IML+zreH2Y083bVowlsyZ8RY1X1qWfY7Y1BDqmxEXDkmHkN2SsC6MDUnE1zhAXnmGKvmbEDkzNR3Frr1BkWNPt"
    "WdZXxXx1ed+GgkIi5WBDCba7/X4x1ImuEc1dmaXdRBNi0jLclmuTolPc31paXqyy2lFN19UZhTTRlx8I3T0ddj7Z2Bja8dGtWde+7ZmWWJzhV4yflu5BdRtf"
    "gqY0p3ryauj+yNNPk0v4TLI4u3Lc4P64mgCXvWkznJmuQUWm297nI1Tg09x33zP8rkvyYzxDa7q4HqRP7Oz+CNc+wysHq3ShboTHeVK9oq6XlyqMc8kVlMS9"
    "PqDb4BYyb+q7DqgiVkK1WYFJlHA+MI4HkzCCTllo03e31E5XfatP+my5rcdtGsb2weY1ofa23lyS6isSzX2JnEQ7I2hN9zL9YF334Zqb98CQzdCK7hxXGm5r"
    "S8r/P8kHGbofpHSa68WkXJQsnGK/B8manVZp6/2Vm6zvMqkBxKbNn/8D7XxlL6gpAAA="
)
_S_SBS7a=(
    "H4sIAMWtc2oC/72ava4luQ2E4zvv4gP9UlRiYHyDk+/ezDAMBwacrAO/f+Cv1JS6B+vZ0FhjvVPT3ZREslikzsevf/l1/OPv718+/1XHx+MPHn/47bd5/iun"
    "+M///Nu+ff/r55+//+37h71GarUn662MlOyff0pdYLbZupVcm8+6wPFqvc2R+uhtuJUbtDpz4wuj+AZncxvd8xxzfTLsfX6kV0opj5mzlVp9DrO6QRtles2t"
    "ld4P6K2n0moezct4gC33nmZqfp6cPjpG+UxrYe/NWnJpy1T2nvOIBdaaevWSWimtxZ5rM5vm1VtryQ/Y+8xd+8itx9u95sxC8zq38tjf17WW0jxbTjqWmms+"
    "YJc1K5xVsg32OoZ1s9JbyzfopeQ5e0p+wDQBrPIXuYxl8L0cyGGX3ApnMWbPsW5W1jhJn7PWvn3FqzWNmt3AxgHTxJ2jJJv3kx1bLCLPNlo+O3zjwfJiby3x"
    "hE/3fkUMYC1GbPgodY6+wV6rsdniE/feYNPTzmLmAQksEzQqvnwYfH/0V/JKZPBEKm7Xt/ur1FQywZcUG9e626sXL5ZLAZklH3C0UTjkOcp+Gzt1FA6k1dbq"
    "w97Xx3zJHWmMOflQvsID0FOuJAVoyiXA1tkZuyuNyMkHnMldj6fc95P4B3/gWd7ux94XHrzCFzcM7CU3J5IPWBNHjGtSPYFeu7ZDvBC91R9gzjMRHi3bBq2N"
    "OhJxPrPnMLhyMI+Rc5u142CfdkFtdP5s5FY1vyD+YNX70AdsQ4bziW28OuJbeIVzMBeNhJk3Ti1KZifTundFEQsDnH02koONFmsLzC+8y3GPTEZy8DfIeXUS"
    "vw6R0Hqdbw2vLDXDN+v1MHjlHo4n+fgHh5NTgVV2kZVh3eIYBzHImqCSZMPywWwMNku2j+tkR8mJU2nkltus2PpavKm/4+z46OTgzVsegZFJ8AO50/P1CVaz"
    "DrlnI+rmwWpRbogLr2WylpqAWBGe9DAVlGk/0vQBlXO4x3svLUAlfoU6CnFv4wYdwoJ6bOz4wJNEEIdrs3gdYfB9/SXHDGdSBgpVwHyDtfJZn3J6bEVPsgD8"
    "xIHj2wNyXBxpJesOlhNrg5+J8rTP8vJbxfcwXJrVKq66MCOLIE2Ok2S+IKhv8AilBt/0g1FFOJaZ2E5grHnynuLZ8jL1uXOtkSYKkcImZpyF6pVzxFlB6T3A"
    "DuHKVaX4jeAbEqcTqW2DSbunakFt08JYOG5RCDtmjW2vWGBJvVQFINvcYIca+DQViFO6weLQqLIg7ddJ9KwP5IHhGgbDcYR+Vj2AL3Fh2WD3onTHAuh5ks8b"
    "RE453JtZIFTE+RkpeV7PilSDXKmIYTCqnWnX+naWB+YGoZfpzSjluDsCIDt5REYlIs/tBhWT1CnFwAZVDOAjcrRGqLyP/4iqxKlCpXB3GGyGW6hvBLm+HiBM"
    "prQaSILuB+uKIUKGRzcIo3AGg7iCQWfY2y5sLC9JrEyR5AFh55qVWGmect7wHfE4ofvjLUBIrFI5YM/AoHN4GQok6oeFvfAgoZJXLZjisV3/YTbIiBJWKDh+"
    "nqySMpOzQLzcYMHL2nvZJ1Eo+/iZTeLZHPbCgWBFbDA5ly28OGAKCSRNdU1jm4NtyGB3KaojTIyEchWgemQbrAizcZycF+fx7TPUZqRLhyeoMhQ6gmODlISm"
    "v0PMnMRS9UECTYWv+wOEZwpiLc39ZNX2BpWqKznD4PEfUd+RdZAXWmR7QP4XA3Cwvk+JxRKtVAGCN+UbRIuJY1Qx9+soI0Gwt7ew9/Oah7Md2jEit80ob1oq"
    "KdLJQLj0BklUtojelIy6CiGKqqiEk8+SHtS8z6felJKRHiZYSIMIMdhzGAdS4DWkx34SuVMoJDqPOm6QEyXIjfqS9+vwgvQd7EO6LIPKwJ/tkDKJsiWI4NvA"
    "SGyFAF7F3APMlBySRuJ0g3wTrUsZ4tGzv5OA+IkKR/ET1/e8xRH6cxACxo7mOE9aITIQa9TXcoPQAoQJoaUDikIILSIu2d6fGgZYBxkFHyAPvYUgJjFgb1i4"
    "86EeIjuLbkxlR985IBVSZNFz2V0ECj3JFeQjubnU3+cfq81G+eyslx5nxmd4kvYLXlyuK3ZAQ6uLCKlCc7/OiZPTQ5J1jGMw5CZHzQbplWzmqIAk/2Iyh1vS"
    "1WhxOBQnxTmC6FIAgiAFcr+gOq6ncEGTRhjrMMLMchx0wIogHJzLzhdE9mIL+eZEbLmeUoyZavzwy49AXSoIMT/N4imtlJfpJpKHlYsvTaWgq8rC9HNDkn9I"
    "/i7OWxgZVMRv2Ifw5sFYu8oaReN6TsTVF9Vl9WBhamUanqqcWuWMVGcuiNBUkwNH9NijSfCpnePrF18siOClinC4FwRG5ClYOPSBmVtY/i95vhpk6ix1BNXm"
    "/VTmQlyZxGbeYm9hfVUg6CWfTxKEJLgYNG17myFNSo+kxmzO4xA7kc06YKQjXNRD8XUVhpkeYNLpUq48HxDh5lA8upmUDHtbo6BeIKYi+TI3I2fNBDhxU6/W"
    "+gar+Ak3UcDtgfXFW4/nkjYoIjOkdZg7/Ph7cSnSVJOO4GpsKDCXEmE/TXLmCdYKi6q53fRIP6C+lkyksvZl7xaYXa539WmnkimNkOOmDR/Jgn8lKaQsih0Q"
    "oqJbIIhHP9WR81LbPjHnJax9nmKLWMPpSSp8Yy5WEh0TcMehmMd3VFjEgd9gVTln4zDfBqvyQvMDSmyY276DzlcSoDx911+KN7YknodqwHly8iECgwK2FfQC"
    "cZY2XY7o1FoRavTNJXvY+4pOiRBURzcV4m1jUFcfnD95ngMjCIk1lzKLcF2YL9JQxgbGkqR2kjrAy3Gqa/4iDMSJVacWvO8vzXD4I+xNQQiMxFV1oVmAz24M"
    "wbZafrPAyDNFOsxKpepBwg9dmYlA9SkKork1Fjw6lRXoOsXhSV7NvxKyBwE+brDCOfh35DvNyWLYCGGJzq9h8OfCpJQlY1ANcNMGVWq1HQig9weYJHhEdbtu"
    "E6lq5skpBGCLwn0py/IiuvHwUBMZU7IFqg/hcRxR9/AHRxLEUkfE9wOjEUL2wKX7bfKYsJYKkFJZJ/oOadlepCOrk7BHJ3nMfvCQmnlKnYYAC6wvYh1xBftM"
    "etn8APEznRp+PU+SPWQAEoMaZw+DR5nwiuK5LXY5wxiyf5GF+sctV9APdK0aOrUjQgBpLlgGp+jbrxlWIyBgTp/b3pu9qKHXuEFcFOUfED5Y8zFWEYtGcSWF"
    "nEjrKv4C2aw0msZ218ALkH0lrZU2Yc752N7Xz3Uejhrq9tUxn9joajj5EJ3LHUXSgkp+/H9AETfKFDZbMoqAeYewROZS8VUVlieuxAKERZEkJJBb7wH6ShqK"
    "qwSOPcGuXh10bJBnWDDnVoffEaMcrC/NUiWETV13iTMpelvfkPbY0UitW6RHCkBDByRNcUprsrxDlHSiyyUEyKjyMCgPPurCnk2qdUYGWDoDS9aFzDc1bMS6"
    "+QHzenENz9uOT1Pbq9njiPljWPtSaKR1nJrD57QTgtygkiLs+s4HHmxrjqCEiOHrBaY817jGfYNLpYteEIF2B8zXRaCwBRsUNdr1lwK1XFKDRVhs0F/wStbw"
    "Uvvs8wF2pR/VMobF46Vxi8TcVAGvD4PXGFPN6ZrAETHXyDLLH0i8qTReZYDioiLt6mnq1ektCOaEUWlzr2IBAyN01tQLj8wwo/Ezcl7dI8qkjpU6JlBicCDF"
    "bFxJZmIbKdAu7V4pOAek7nOUKGlWFeDQRHvqMchbm7UwuKqe+hWKPt3wct+COnxAWWcP6Op8PQV7sV0CR5y3IfW7cGGv1+hACSt/8z96roKZh9CUAF2jDd2k"
    "bGE0h6YJSdvdczeBLnFHTHV8doM0LoQrYTYPSM+OayerUCv+/kFoovq1L/UUs5xpGD4gHmAy1P08UxMyVhcvUjL1AZKdJnr0M19xTtdc0pRwDYN7lkIbv9S6"
    "JmatHLBq2JrUMW2NpTEKTMHy4Zk9IhWosaCmyl4OqGsiTqEq4FsYPMMU0yRCk/9a9+STN5AVcCE0CysHqDkYbm7UHD8KdA3HvEp0zHyezPqErgJoVS8XHrFJ"
    "JVW5dok1wnODGnsPXcC5laPHkPPq1DSt7g+sqg4UTZtoHfk/jhj9iZIbO8UvuTleeG8NhSu17LoX0mWaJsDyiU5pix8qqXSBWtPRb1BdKoFMTU1bTqEC4WhH"
    "u/Ovh733uoeSbMbbhHxq9+UUb+hubPpp0Ck1It6pIbvPG+x53aBApKeVb2u+SS/vd0l4zDN9dXKmeyz4oW5QEyCEwJIOcXiaIGcto6n1LTcobYxyHGkHF90u"
    "gk5NrAYwy55q3kS0dIVnER3XLSZp2TUCndq07WMiHsSCPFr3sGGBTfNXmKj6ebJr/EAnVWnc7g3eurOgv4k4RCPM2Y8UyRpwEXh87gxJ1FRo87pk9AdIHiaN"
    "OXe6EtVEn67G1lwsDEYSqkjmdR86VVMPSAlAIFEcCN4AdW+X1LcTqfsaTGCaWfczUMQmJAJDAy8aYkmlMLhdWDVHLFJzSKtYIkaKhos0Axi2DapBUi9QdN99"
    "g02T4KEp0wFRdGgqXZWwy29fz5kmYQ3hqmRWTT43SJ1RCCUFUrmHbTCmr6npnvIvsEhNrHpwutahGRfaQ8PDMLjvg8jZrEtR6uvcQ15ATVeGbiVHPvdBumoi"
    "wTWT3mMDgRLEKLd5z+zRN4gsoouNUym+QnlWOhnNusmvoRuR0B49SWq5BoR5BkafpaZoJbm1G6ySiEOvb5lFDvNRiBTWueR22Du3CkSU65IEQtrETZWjcFM0"
    "ibGThepe6E50r6DUv0FNm8oaltV7g0kXcUlisi6DfzTTXGV/Tbrulgdxb1Jriv3UnmCS5LADsWNVRdxU2zWz/Qrh2ZE6mtoTu12UGVfS4kSN3KUm44K5vwrU"
    "R5EY1zD2BtdcfFXYeu7D87plUOReojEMvvUzhr6Ig++io/PSIPZSpz4kYvX7hha6BubRRYPrYrGNG/QKcalIzxqvm8bITQMERNACw+D+FQRE6hoz4rDjF430"
    "lIb6icVJQs5CXS9BMqXoH+CiDk179ghDbY5CCQK6AvQaaeo6QrKCiJ6XEgGxiJWZl34oy2e6Oyqaph8EfVCgxDUtLkU1RBdXUrIeFpRzGhytkRzZfE1whEhu"
    "Kcv7yiFOSOMszs7zdaFWNe9Uk0eDWxe9IPYpeIR0Vte/97CIcvxYXRekMeJQw1jEfYJsTY00qViBERDn7IoJgnXl/VhOob65Llv2RuQZ3klr1reUTbsQyR7K"
    "oparo8q61oLb4TLRVCBJdD800xkLabrUGBoS4iksPHSlqEU1EW4RxWww6RqR5kBncQSPxpKaf+gHMvmAYovVLNtmZb4HNxW0DmHi2+DnHtJhSJfxJm/ub9NY"
    "1zVV4KgP1+riXZd3WTd9foMai2pckc7SSEj9MAHRDTtbGPz5LAUvkfWXvNyg7kabbvkymX2D6lBKlmQdvkH93kfTSN0oeRDGQ1hK8K5ba7HE/gWAfvSEPYJK"
    "1++bEGkaslQyauxMBBeIOcocCv4uA1TKkVYD23wZ3MKS1jhrmKoJCscVmH7q0DRmwxH1DNk497LWEHkqTMSp+7jme2gnpl+UwApK2Ar30S1XiWYJbdsl3tZF"
    "QNegSHfRG5QUX/ex42Z7ZSA4JWoc8aJHskZ96Bq8EAa3JMmzrDtkXU/YDg0y03TdWHTNcfoCfYVdS2f5A9QPhzR/G+Xcmereldqpu0zfBo+q1PimaHlEnx1w"
    "hNjM52bSRD7asqmizxskYAZta7XzAw51XdQ36TYL973vXxQh3FYOHkWhXk8/jyIH+7lq14SSho++Re2W3+C6H2cZN5Q0uENckDhhayvKrMGI9I9kzZljRf8m"
    "SVPO7F5995jrXvmeeGnk39UHoWfPumBBtQaKod7C4P85+1TMXD9ZqhLpRVdtY7ctnJruhjXw9bEv+pRHhLJukFO+QalQFImu+gIk2ps6LF2T1Gs4/F+e+hhm"
    "zikAAA=="
)
_S_SBS1=(
    "H4sIAMWtc2oC/72aOw8ktxGE47v/ogWfTTIxcN5gc2kzw3AkwIkc+P8H/orTzVkLdxcKMqRz386Q7EdVdXO+/Pb33/K/Xr8+/13Hl/vP8/rzH3+s+ENO15/+"
    "+x/7+u0fz799++e3L+mRUprTch+lpjlGG24blkqZlmq3ZH0bc+4rt5pXSb2MYyol9756qmNdtpTHzGut1gYv9LWe++94ZWqpWS55pdrcVnjP6jPN5q8tNkqf"
    "ZZm1lqwe22o5zZStZLtstZe56uIv+F33pV7XEfIYqWaesMmeRxjXKIMD2FjT35Ly7D2XXtnW4Cy3cZ+lpNZqGDmtGf+qwzizL/i+zs3Zil6QWNcfyMU6zrI6"
    "ZsvzOkhuc+XUR+vsevZjqynVMvPkdP7sGHWOVkvB72Ov9fKYZUWGB8as1qeb6kq9DatlEM3LP63WRmxraqvVdWz4urSVRg4/5mqD53ATb5m+1NO3Ufh7jlHZ"
    "ZfYj5DkI2M6ZEidgO8bijbOk21b6WM0a/zV/XV+JDOAf/BtLRchwHu4mxiPzozDaYGe5EuWxImSEsCacM3NuZd7GUlmA4ODcMLbF09Nmx9u+nkcssy+SKFla"
    "PUdGE+A5iFofY7qtFiuVY7U1S7gAGxvNiT2RuP4saTFqnsqdWfZa7ytipSelkfU6shdZaSPPNuZVMftchCCRb8kG273KSSbSPi/W5qz7wYk7OgVCcbaZfZkr"
    "WgSAE5DEpOcYVxio7pZyt7mPeiXXYO8lKxfKyr2GTaWdFhVlnqzWi4pl9pHI1+FrKVycJRkVXhtV2/XWOhJumovtTZwjC9nB2mABYSnLLZXKJ8lVcLIsa/xc"
    "rqEEYoUrQORYHkIj3NwdLsimSanVarWNuj2ZCSSJox/X1FeY9F5ScNg0f5sRA+tWgQVT6r0/QFDpmpeRKax6lYrKC5+zN+rVHQda4hFcyr4TRXsbC1vE04Qr"
    "nlaC8zDRIoOLrxdAiBcWlbkEoV6Z4KAAqyvkXi4kM2CxdD5grhxbJxp4H7+VeBYI7qbiHrX6Wq+fpDnANnWqyssipVtRvfYOrHruyMaWWLICICVAf87a1wRC"
    "1gb49wFBwRA52wYbJBPXQV2qXdtWEtf4JftKymsYZM7bSMiF+xBKGPuYgM2aywCXvd4zgDAtQJm9gTRktNsAdzmCeDrogQyUFdsmYbOD47ZVUr6RnB7ITHHh"
    "lTxI0m6+0hUxMBTIWeQURdK62wg9CVnWSMVfy7IgwCaa2Vo+NqoROluCaX827zxKbTag1NcKICRW+ESRH8EHGCtuEG6TT5Fk1DDrTKqWgkjzNi78TQJnwCUe"
    "Zwt4Fx+mUX29K2qQURaniEw9GTr/Z4p42EJ1igEHCDoJQxSHx7abiJzwZiDDWbaBTpXTiVhKvxI/mGsTahVOV2jKihtBGDBNOEjyH1rmnNBfZW99lQ/jLAPQ"
    "B5Tjl4Lnqu0BCcl8wStuBBk+BpZJSMB2hJEEgb+IoBznRhLdRB9ABhh6jFU6QuBhLZ7G3yikBfpDVb6ex64AtOR/1z57JLv2CqyAjkvS4zIKLhaJIe1SHXBl"
    "7BRzgchxkh+Q5OLAiB9qO1scMEoOV2dQl/MbiRbRXlMEScKSDZECZAME1kDAVeLUMipaqkz+6st6QONJxNSLtvH7L6l/fX4KRv0akJCEoiD83cRUp+TdnNrz"
    "EJCgshalQH5ExlZhCfKPnIFD5/10l1AiN1ss5wIEgq0khoRoHq6NhogETIBDQRAXMUJ1ggQS27JjI/LsdOjhEEpUfDOoDVwtvtaLc1fpHJ0o6dU6N0bSEplW"
    "FHwAaBvnA5xVCRBSweUxQrPU/BT/mxuFxrPtLBvX477g+0t5/D8J/P4Lcqw88txoLgRlK24Ur+NLYJfz2G3kxGhEvFGrG8Wv5AHbAsanjHtB1V95cDTYWtmN"
    "cGx7i+VRxQLwV8V7dbkRrCATjd8OnHcbKVSFo0Fs9+P6n3oIaO6cUPU3HnB1BR5ZVbJmPzEe4B0RB1fA2jLdz0kqFUFCKpObt7FIiylpWvPHpwGtsHRDqNT6"
    "seArEgxgxTPEg0i7Da4Ur1OScq8bi2QoGbemmqAPI9GiBpBykdzoGqEMyMO/fbm/OIKuJEVkIhTUwGJdF/+gPe0SVYYyd/WtrYEV1OkszowySTThBXTQOo+S"
    "360Lh1IsFdiJ+oJKKCy1KONgmYoCRcsGegtMwLlo/yHXrXob51bktDxR/+QfrVwRR/O33RdU7NBJ2p82kzfDwt/43cBvCW8tjz6A00Sw+NgsLMQQXsBneVsS"
    "fgbjBPqp+AIBln9eY9sIKvKeIhAy+DYrQIkehK6mRVssI05P+k+vQcFLCwKetLzQI+tdurI91KGJleFMxMNO3/bg6CQjCh3gzZexP6AbUEckRlLn26g2UwEE"
    "w/3xJq1MWaK+wR/P/ltYJqlods1BBFK+b/WR4BK0okJyzG27xvVfAlbWbWzE1XTY5jUhdTA6GYpDQDdfMMqtNAkI0r0oEifyTaQmBT9PvQkcif3cPLGOkQfp"
    "qZe6mtgvkCEtQjIre3zBd+SfGkgJ12kB6tKtOJYOrEyWvXdBTqhxQ4rOdRt5O7p+a5xISg7XFCkcPm0v+NyQ+f0K3w0lW1MQmxvVIo2xJRrUdhulF4mHZhxR"
    "4dQc/aQGCcu6V/gtNb/TLCJDgCXcMUVR3ouwM/qKpcnLiBKXDT2oyqFz8AkLSgXf7EPiG18rwmfqGjUDQvrFRIQ/kmBoL05pIb2aBifsp091rus2wvnkM9i9"
    "gs1xbVct6I8l+4JvJb+pLaLs0QB9REVkzQLYMaq310j+ollP0Y8hrdtomqaol8gzKoJuWV0eutR6PRUhxqsPXgz/IFjB7nkVVn0APEo3eQQk8NcgQVBjGlbs"
    "OLvNVFBZTHOelsiSyJMCLB/LBWaijqpUO1Bfgn+gA7Sy3o0pZKt6LTBDqAOW3kb1aVqAoEd6Fk2SaGlo20hEXzCahe8MnjRnASebJJIGONFIqbtVXmSkbztG"
    "xJiEE0h15mR1NyGEgDDV4gsGho6sPGsaB0ajCDvSq0gHqFU/VMFJIBvgpGuecoy2mwTqu/e7RRzCymxgOTT0+hCcWZwNPEMilJbPWgDQYmooCbqfL9Pc0f5u"
    "mZwdX2TrmgyQjNFu6MSoiqYq5D++VuAn8G5qOJrwYB08IgmyBmbp8KC8S1KR4gD6UQqCWdoZoXMJb8IO4jvUqpRl9QU9fFhIcPZLUs6YMvQpmZbLRgov7T2a"
    "oTueUtpw4G2k863C9xlDL3VjyNOqI1JwvuBPBEsnkBWeo1pmaJOO3+lWwDig7jYi8hBwO5MDzoSjwDyBJiE2nL1ccq6HZpd7mGpUi0pIHUbT+AIhNMQElzEj"
    "9nreKpvurl0VuI2UAkgoQAytWKQTJZuatKce9wWfkYtNPicy/MiiSQRz8m7BAeh5spZ4AGrQkuTWbWzoClVKtfhlo4Sn5r5K7+oLRgnqcEtuwX/tbEKSgpJF"
    "0I2owEatTw0OszjhGEVPRRW86jHK79R261REHPAvDmFoTtXCmGTw2lh02aqaUSJAXlcHKlQAUhY9MRCI89joRMEn2oPmBN/U5asounjKl3oqsvAbPNc043LJ"
    "QxdA2eN76gpfRA5o1I0HyCQq69gmnaJ6WykmbxcQM+oI6laxF16/juQEpcEVSM5Ugm1bjFZbMkCTtC0naduFvyDAxbyy0GEKoaXstlCVjB6qN6GYL/CGLIg1"
    "UdzDsOQ0Vh8a0gMZwoJcsnONRrY0lOgH6mAco2lSxwboxrs/LqkrIiSNRvYDXZJzPngfsVrgPohXwwWa0MCR0HrL7iv6AJSCftZzWcdIJo60C8U7s/mgQjVB"
    "1jCo2cdyUWwkRSmkL/LhSD25S+5GgLd74qeSJ1rQmLVPI4ir6Xw78pGFTKNfRCwlOHzBGLDw4rYZRZcL0VU0ILpoas7mgwShdOKSJZXHuCcNunwCT7Z2C0m5"
    "dm9Lz8BPfbmQm7BPE4shkyzGK5KbQAYhTar8eDVER3GlrWPaMWosQJa2QwfobUBA2dyBlL3cPdDsmsTBzurKW0wlTaSouWSKyyGJRGp4qSb7sXGk/fIcF0s8"
    "ZaaSIsnL8KV8sLLEdqrLWXqZ52JpaUhQdJFzbpHUr+yBUh12bLoQSNLSMWshLEVM2zRY8aViBD00h5MSq8sjQVJ1ZZZpAOoHJRfByrXBceZjE2Rpeg9sxVUb"
    "bl1tF2GNU73JdGhRQklxKV5ZGNWLm27dqK/mWU1FcEAEYAE41zF2uUJgOF0EDsgKv3ZJqdrKXQCvn7QHHAFRihPnrGcAQKoPtYkSnes2Lo11Qbhhpz3QHR7l"
    "RtLZciy+BOYPwR9Bl/fImVXdSCS7RqWa15z1ME48B1GAsQH+qCDbJVVTvbjDF3wdTaMK2ndKbZyJCrWSNHWUDo5GccOvKp7f13EbAZuiAo/6QU2T42rxaFTn"
    "8PXeEL5Qgy1rLppdA9AdXPBAKa7iRtahm2m68xnrtnXdyjRNXuOHvVTJd7omnHrB5dvFpT26+kCIRIlRL9DFmEztLCprRgrpagy07uQXNBBG2uKkix5O44Mz"
    "e4gp2743s/KxWHR2FS6dG2yKT+sLa2lksVtEiy4OiUwCmGkoN46taeAoeXvuySk5kJedgDi+1OtHaVLEgXOY5umna1Wrl0AI8hAWPEaFJWuwdycwvIqYKNLc"
    "6JqdJm/Xlf2hy4qm4VKXGNiKDmPVhaL0B37eOg2vAb9kjdp1darHWDRsycpXvRsjrqRZ5fSIOerStsx7n1Hm908IeqrD0QQtxb4BRSlmYRhi8jaqfQdp0V3z"
    "1JG+TyiK+yzrnPDjKkEVIgISwAbzqMklUJSWX4RuOpryKC6d99xMQJfFaUObjG4RTOGIFVuz4uu9OP1Sk5bVCOha0rNrmZpvAgE85UhODlJ0t6Z8rPMYCSrO"
    "1LwnnTQGMqWbNOdK9aTnuRDvOoMuclZ8MaErChhiTeWhN660w2rAbd8l5XJsHAPRB/L61ULe44qtpXTVspcKRalbB51risji24DdUid9MnF0sK7vKWiQjX4m"
    "9iTuq0XCCrETX51Mk3wu8ljxpZ7SWvTwliUK6C1GtPDsC4yCQOHpdbQWPITKG5pH3EZqHvqHFFYO/QY4KQ8AbbUCx40+xcx7qCq6o3zWZZHIIiPUIO/BJsvD"
    "6KgqXZZkt+hWG6Fr1xgsS5p2XWGyoRLe+2EDoE2zSXQV0DUC7tU8NXXr49T21L3H0pBKo2A30oHo1grZDglEbd/X4zY0vhLO6N89Pg3Yn8ygfcvwrwWWxL+m"
    "OVKn49hYr+kqsrpSMdGzrkrQFLzRl3r+GLfUrSw1RK3ebUzSRwlCh/zR8EAcScjdclQ6iZKF3rx15PVxNOGkJDw1RdoQmPuaAy8qcfc3Fx5z2JWgopen2CSM"
    "OK2pGaB7KOdxyEY3J2jYdOWRL/j2q3rd5auElVc1voESgpel2UF88IQv8BH/4OQ6jw03NI2gWqhDTUU4G3m6NO5+f6hJzc/IPfBB0iPmCIRGhdslzs+sVNce"
    "SGhCUlWRtxGCUKdHRALb9g24dCNSy6N3bsm/Mz3BpjE6Z2N/PjRXf46a6rpSyT6+kU24pi8riLPfnJeNEUPsmGKtV4ylTdxb6r7WCbWhz19M39FYOTetSAek"
    "tiawUuj1GKUNkmkOFwImqQxZHUkzPFPillwfGQCO+nLmTBWaaFvTMuxrRHOr6bDqDows8bslGQIAIIS7k7ypPEmRoe/O2l7rZ6LSNs2rRZvjsDf0kUR3YN9Y"
    "H0YTuHKEFqJSTXzWwEWFZKcMxG0dNgR6iUQR7i4ftkLUVQ1R15cBQUCiAxsbUvI8RvUi+vBFsn3442PfpGm2pXHvKYPXPfeqWV8CaZi/IiZqn+GwJgxMITXV"
    "7eqjEzRaOwyLsWgugKzr50sR8BY3aSSR973B22XlD+WC+lR9UJXSKgdIhe00O7rTmbeRQhDbEa3LpZk+fOiDGqBulSsiX/8HDWKkxo4pAAA="
)
_F=(
    "..."
    "/gMQ79beOgYIAA=="
)
os.makedirs('prof',exist_ok=True)
SIG_B64={"SBS17a":base64.b64decode(_S_SBS17a),
"SBS2":base64.b64decode(_S_SBS2),
"SBS17b":base64.b64decode(_S_SBS17b),
"SBS7a":base64.b64decode(_S_SBS7a),
"SBS1":base64.b64decode(_S_SBS1),}
for k,v in SIG_B64.items(): open(f'prof/{k}.txt','wb').write(gzip.decompress(v))
open('prot.fasta','wb').write(gzip.decompress(base64.b64decode(_F)))
SIGS=list(SIG_B64); print('signatures:',SIGS,'| CDS',open('prot.fasta').read().count('>'))


### 3. Helpers (codon translation + signature application)


In [ ]:
import random
CODON={};_b='TCAG';_a='FFLLSSSSYY**CC*WLLLLPPPPHHQQRRRRIIIMTTTTNNKKSSRRVVVVAAAADDEEGGGG';_i=0
for a in _b:
    for b in _b:
        for c in _b: CODON[a+b+c]=_a[_i]; _i+=1
def translate(nt):
    nt=nt.upper().replace(' ',''); aa=''.join(CODON.get(nt[i:i+3],'X') for i in range(0,len(nt)-2,3))
    return aa.split('*')[0] if '*' in aa else aa
def load_profile(p):
    s={}
    for ln in open(p):
        q=ln.split()
        if len(q)>=2 and len(q[0])>=7 and q[0][1]=='[' and q[0][5]==']':
            try: s[q[0]]=float(q[1])
            except: pass
    return s
def mutate(nt,prof):
    s=list(nt.upper())
    for i in range(len(s)-2):
        ctx=''.join(s[i:i+3])
        for sig,pr in prof.items():
            if ctx==sig[0]+sig[2]+sig[6] and random.random()<pr: s[i+1]=sig[4]; break
    return ''.join(s)
def accumulate(nt,prof,r):
    c=nt
    for _ in range(r): c=mutate(c,prof)
    return c
def load_cds(fa,maxaa):
    out=[];seq=''
    def flush():
        nonlocal seq
        if seq:
            cds=seq.upper().replace(' ','')
            if len(cds)>=30:
                aa=translate(cds)
                if 20<=len(aa)<=maxaa: out.append((cds,aa))
        seq=''
    for ln in open(fa):
        if ln.startswith('>'): flush()
        else: seq+=ln.strip()
    flush(); return out
print('helpers ready')


### 4. ESM-2 650M + pooled embedding (fp16)


In [ ]:
from transformers import AutoTokenizer, EsmModel
dev='cuda'; MAXA=380
tok=AutoTokenizer.from_pretrained('facebook/esm2_t33_650M_UR50D')
emb=EsmModel.from_pretrained('facebook/esm2_t33_650M_UR50D').eval().half().to(dev)
D=emb.config.hidden_size; print('dim',D)
@torch.no_grad()
def embed(seqs,bs=8):
    out=[]
    for s in range(0,len(seqs),bs):
        ch=[x[:MAXA] for x in seqs[s:s+bs]]
        e=tok(ch,return_tensors='pt',padding=True,add_special_tokens=True).to(dev)
        h=emb(**e).last_hidden_state; m=e['attention_mask']
        for i in range(h.shape[0]):
            v=int(m[i].sum())-2
            out.append(h[i,1:v+1].mean(0).float().cpu().numpy() if v>0 else np.zeros(D,np.float32))
    return np.array(out,dtype=np.float32)
print('embedder ready')


### 5. Embed WT (once) + N mutant draws per signature


In [ ]:
N_DRAWS=4; ROUNDS=8; N_PROT=400; random.seed(0)
prots=load_cds('prot.fasta',1022)[:N_PROT]
print(len(prots),'proteins; embedding WT...')
WTe=embed([aa for _,aa in prots])
profs={k:load_profile(f'prof/{k}.txt') for k in SIGS}
data={}   # sig -> (WT[N,D], MUT[N,D], PID[N])
for k in SIGS:
    Wl=[];Ml=[];Pl=[]
    for d in range(N_DRAWS):
        muts=[translate(accumulate(cds,profs[k],ROUNDS)) or 'A' for cds,_ in prots]
        me=embed(muts)
        for i in range(len(prots)): Wl.append(WTe[i]); Ml.append(me[i]); Pl.append(i)
    data[k]=(np.array(Wl,np.float32),np.array(Ml,np.float32),np.array(Pl))
    print('  ',k,'embedded')
print('done; per-sig pairs:',{k:len(v[0]) for k,v in data.items()})


### 6. Fit the operator per signature; confirm affine ≻ vector at 650M


In [ ]:
def split(WT,MUT,PID,frac=0.75,seed=0):
    ids=np.unique(PID); rng=np.random.RandomState(seed); rng.shuffle(ids); tr=set(ids[:int(len(ids)*frac)])
    trm=np.array([p in tr for p in PID])
    wte=[];mte=[]
    for p in ids:
        if p in tr: continue
        s=PID==p; wte.append(WT[s].mean(0)); mte.append(MUT[s].mean(0))
    return WT[trm],MUT[trm],np.array(wte),np.array(mte)
def fit_operator(Xtr,Ytr,lam=1.0):
    Xb=np.hstack([Xtr,np.ones((len(Xtr),1),np.float32)])
    A=Xb.T@Xb+lam*np.eye(Xb.shape[1],dtype=np.float32)
    Wb=np.linalg.solve(A,Xb.T@Ytr); return Wb[:-1],Wb[-1]   # W (DxD), b (D)
def metr(Dp,Dt):
    fc=float(np.mean(np.sum(Dp*Dt,1)/(np.linalg.norm(Dp,axis=1)*np.linalg.norm(Dt,axis=1)+1e-12)))
    ev=float(1-np.sum((Dt-Dp)**2)/np.sum((Dt-Dt.mean(0))**2)); return fc,ev
Ws={}; res={}
print(f'{"sig":8s}  {"vector faith/EV":>16s}   {"operator faith/EV":>18s}')
for k in SIGS:
    WT,MUT,PID=data[k]; Xtr,Ytr_full,wte,mte=split(WT,MUT,PID)
    Ytr=Ytr_full-Xtr; Dt=mte-wte
    v=Ytr.mean(0); fcv,evv=metr(np.tile(v,(len(wte),1)),Dt)
    W,b=fit_operator(Xtr,Ytr); Dp=wte@W+b; fco,evo=metr(Dp,Dt)
    Ws[k]=(W,b); res[k]=dict(vec_faith=fcv,vec_ev=evv,op_faith=fco,op_ev=evo)
    print(f'{k:8s}  {fcv:6.3f}/{evv:+.3f}      {fco:6.3f}/{evo:+.3f}')
print('\nOperator should beat the constant vector on faith AND EV for every signature.')


### 7. Spectral interpretation: SVD of each operator + cross-signature subspace overlap


In [ ]:
import matplotlib; import matplotlib.pyplot as plt
K=10   # top directions
US={}; svs={}
for k in SIGS:
    U,S,Vt=np.linalg.svd(Ws[k][0],full_matrices=False)
    svs[k]=S; US[k]=Vt[:K]   # top-K right singular vectors = input directions the process acts on
# effective rank (participation ratio) of each operator
for k in SIGS:
    s=svs[k]; pr=(s.sum()**2)/(np.sum(s**2)+1e-12)
    print(f'{k:8s} top-5 singular values {np.round(s[:5],2)}  eff.rank~{pr:.1f}')
# cross-signature top-K subspace overlap (mean squared cosine of principal vectors)
print('\nsubspace overlap (top-%d input directions):'%K)
print('        '+'  '.join(f'{k:>7s}' for k in SIGS))
ov=np.zeros((len(SIGS),len(SIGS)))
for i,a in enumerate(SIGS):
    row=[]
    for j,c in enumerate(SIGS):
        M=US[a]@US[c].T; ov[i,j]=float(np.mean(np.linalg.svd(M,compute_uv=False)**2)); row.append(f'{ov[i,j]:7.2f}')
    print(f'{a:8s}'+'  '.join(row))
fig,ax=plt.subplots(1,2,figsize=(9,3.3))
for k in SIGS: ax[0].plot(svs[k][:30],label=k)
ax[0].set_title('operator SVD spectra'); ax[0].set_xlabel('index'); ax[0].set_ylabel('singular value'); ax[0].legend(fontsize=7)
im=ax[1].imshow(ov,vmin=0,vmax=1,cmap='viridis'); ax[1].set_xticks(range(len(SIGS)));ax[1].set_yticks(range(len(SIGS)))
ax[1].set_xticklabels(SIGS,rotation=45,fontsize=7); ax[1].set_yticklabels(SIGS,fontsize=7)
ax[1].set_title('top-%d subspace overlap'%K); fig.colorbar(im,ax=ax[1]); fig.tight_layout(); fig.savefig('operator_spectra.png',dpi=160)
print('saved operator_spectra.png')


### 8. Operator composition — does $W_B\!\circ\!W_A$ predict applying both signatures?


In [ ]:
# combined mutants: apply SBS17a THEN SBS2 to each protein; predict with composed affine operators
A,B='SBS17a','SBS2'; random.seed(1)
comb=[translate(accumulate(accumulate(cds,profs[A],ROUNDS),profs[B],ROUNDS)) or 'A' for cds,_ in prots]
Ce=embed(comb)
# held-out proteins (same split seed)
ids=np.unique(data[A][2]); rng=np.random.RandomState(0); rng.shuffle(ids); tr=set(ids[:int(len(ids)*0.75)])
te=[p for p in ids if p not in tr]
x0=WTe[te]; ctrue=Ce[te]-x0
WA,bA=Ws[A]; WB,bB=Ws[B]
# affine: mutant = (I+W)x+b ; compose A then B
def apply_op(x,W,b): return x + x@W + b
pred=apply_op(apply_op(x0,WA,bA),WB,bB) - x0
# baselines: single operators, and sum of vectors
def m(Dp): fc=float(np.mean(np.sum(Dp*ctrue,1)/(np.linalg.norm(Dp,axis=1)*np.linalg.norm(ctrue,axis=1)+1e-12)));ev=float(1-np.sum((ctrue-Dp)**2)/np.sum((ctrue-ctrue.mean(0))**2));return fc,ev
print('combined SBS17a→SBS2 prediction (faith / EV):')
print('  composed operators W_B∘W_A: %.3f / %+.3f'%m(pred))
print('  vector sum (v_A+v_B):       %.3f / %+.3f'%m(np.tile(data[A][1].mean(0)-data[A][0].mean(0)+data[B][1].mean(0)-data[B][0].mean(0),(len(te),1))))
print('  operator A only:            %.3f / %+.3f'%m(apply_op(x0,WA,bA)-x0))


### 9. Save results


In [ ]:
import json
json.dump({'per_signature':res,
           'eff_rank':{k:float((svs[k].sum()**2)/(np.sum(svs[k]**2)+1e-12)) for k in SIGS},
           'subspace_overlap':{a:{c:float(ov[i,j]) for j,c in enumerate(SIGS)} for i,a in enumerate(SIGS)}},
          open('operator_results.json','w'),indent=2)
print('saved operator_results.json + operator_spectra.png — send both back')
